In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install kagglehub

In [ ]:
!pip -q install torchmetrics

In [ ]:
import kagglehub
import os
import re
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import random
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import time
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from torchmetrics.classification import MultilabelAUROC
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
DATA_ROOT = kagglehub.dataset_download("ashery/chexpert")

train_df = pd.read_csv(f"{DATA_ROOT}/train.csv")
valid_df = pd.read_csv(f"{DATA_ROOT}/valid.csv")

def build_full_path(rel_path: str) -> str:
    corrected = rel_path.replace("CheXpert-v1.0-small/", "")
    return os.path.join(DATA_ROOT, corrected)

train_df["FullPath"] = train_df["Path"].apply(build_full_path)
valid_df["FullPath"] = valid_df["Path"].apply(build_full_path)



In [ ]:

train_df = train_df[train_df["Frontal/Lateral"] == "Frontal"].reset_index(drop=True)
valid_df = valid_df[valid_df["Frontal/Lateral"] == "Frontal"].reset_index(drop=True)


In [ ]:

SEED = 42

def extract_patient_id(path: str) -> str:

    m = re.search(r"(patient\d+)", str(path))
    if not m:
        raise ValueError(f"Could not parse patient id from Path: {path}")
    return m.group(1)


train_df = train_df.copy()
valid_df = valid_df.copy()

train_df["patient_id"] = train_df["Path"].apply(extract_patient_id)
valid_df["patient_id"] = valid_df["Path"].apply(extract_patient_id)

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(gss.split(train_df, groups=train_df["patient_id"]))

train_df_train = train_df.iloc[train_idx].reset_index(drop=True)
train_df_val   = train_df.iloc[val_idx].reset_index(drop=True)


test_df = valid_df.reset_index(drop=True)


In [ ]:
SPLIT_DIR = "/content/drive/MyDrive/FYP_CheXpert/splits_fyp"
os.makedirs(SPLIT_DIR, exist_ok=True)

train_df_train.to_csv(os.path.join(SPLIT_DIR, "train_df_train.csv"), index=False)
train_df_val.to_csv(os.path.join(SPLIT_DIR, "train_df_val.csv"), index=False)
test_df.to_csv(os.path.join(SPLIT_DIR, "test_df.csv"), index=False)

print("Saved split CSVs to:", SPLIT_DIR)

In [ ]:
def summarize(df, name):
    print(f"{name}: {len(df):,} images | {df['patient_id'].nunique():,} patients")

summarize(train_df_train, "train")
summarize(train_df_val,   "val")
summarize(test_df,        "test")

In [ ]:
TARGET_LABELS = ["Lung Opacity", "Pleural Effusion", "Edema", "Cardiomegaly", "Pneumothorax"]

def make_targets_and_mask(df: pd.DataFrame, target_labels):
    y_raw = df[target_labels].to_numpy()

    mask = np.isin(y_raw, [0, 1]).astype(np.float32)

    y = (y_raw == 1).astype(np.float32)

    return y, mask

y_train, m_train = make_targets_and_mask(train_df_train, TARGET_LABELS)
y_val,   m_val   = make_targets_and_mask(train_df_val,   TARGET_LABELS)
y_test,  m_test  = make_targets_and_mask(test_df,        TARGET_LABELS)

print("Train valid rate:", m_train.mean(axis=0))
print("Val valid rate:  ", m_val.mean(axis=0))
print("Test valid rate: ", m_test.mean(axis=0))

In [ ]:
def build_full_path(rel_path: str) -> str:

    corrected = rel_path.replace("CheXpert-v1.0-small/", "")
    return os.path.join(DATA_ROOT, corrected)

for df in [train_df_train, train_df_val, test_df]:
    df["FullPath"] = df["Path"].apply(build_full_path)

for p in train_df_train["FullPath"].head(5).tolist():
    print(p, "exists:", os.path.exists(p))

In [ ]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [ ]:
class CheXpertDatasetMasked(Dataset):

    def __init__(self, df: pd.DataFrame, y: np.ndarray, m: np.ndarray, transform=None, path_col: str="FullPath"):
        self.df = df.reset_index(drop=True)
        self.y = y
        self.m = m
        self.transform = transform
        self.path_col = path_col

        assert len(self.df) == len(self.y) == len(self.m)
        assert self.path_col in self.df.columns, f"Missing column {self.path_col}. Available: {list(self.df.columns)}"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, self.path_col]

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            x = self.transform(img)
        else:
            x = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        y = torch.from_numpy(self.y[idx]).float()
        m = torch.from_numpy(self.m[idx]).float()
        return x, y, m

In [ ]:
train_ds = CheXpertDatasetMasked(train_df_train, y_train, m_train, transform=train_transform, path_col="FullPath")
val_ds   = CheXpertDatasetMasked(train_df_val,   y_val,   m_val,   transform=valid_transform, path_col="FullPath")
test_ds  = CheXpertDatasetMasked(test_df,        y_test,  m_test,  transform=valid_transform, path_col="FullPath")

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class ChexpertDenseNet121(nn.Module):
    def __init__(self, num_labels=5, pretrained=True, freeze_backbone=True):
        super().__init__()
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        self.backbone = models.densenet121(weights=weights)
        self.backbone.classifier = nn.Linear(self.backbone.classifier.in_features, num_labels)

        if freeze_backbone:
            for p in self.backbone.features.parameters():
                p.requires_grad = False
            for p in self.backbone.classifier.parameters():
                p.requires_grad = True

    def forward(self, x):
        return self.backbone(x)

In [ ]:
NUM_LABELS = len(TARGET_LABELS)

model = ChexpertDenseNet121(num_labels=NUM_LABELS, pretrained=True, freeze_backbone=True).to(device)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
CKPT_PATH = "/content/baseline_v1_masked_frozen.pt"

model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
print("Loaded:", CKPT_PATH)

In [ ]:

def set_trainable_v2(model):

    for p in model.parameters():
        p.requires_grad = False


    for p in model.backbone.classifier.parameters():
        p.requires_grad = True


    for p in model.backbone.features.transition3.parameters():
        p.requires_grad = True
    for p in model.backbone.features.denseblock4.parameters():
        p.requires_grad = True
    for p in model.backbone.features.norm5.parameters():
        p.requires_grad = True

set_trainable_v2(model)


trainable_names = [n for n,p in model.named_parameters() if p.requires_grad]
print("Trainable tensors:", len(trainable_names))
print("First trainable names:")
for n in trainable_names[:25]:
    print(" -", n)

print("\nTrainable param count:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
pos = (y_train * m_train).sum(axis=0)
neg = ((1 - y_train) * m_train).sum(axis=0)

pos_weight = torch.tensor(neg / (pos + 1e-6), dtype=torch.float32, device=device)
print("pos_weight:", pos_weight.detach().cpu().numpy())

In [ ]:
def masked_bce_with_logits(logits, targets, mask, pos_weight=None):
    loss = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    loss = loss * mask
    denom = mask.sum().clamp_min(1.0)
    return loss.sum() / denom

In [ ]:
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4

head_params = list(model.backbone.classifier.parameters())

backbone_params = []
backbone_params += list(model.backbone.features.transition3.parameters())
backbone_params += list(model.backbone.features.denseblock4.parameters())
backbone_params += list(model.backbone.features.norm5.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params, "lr": LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY
)

print("Optimizer groups:")
print(" - backbone lr:", optimizer.param_groups[0]["lr"], "n_params:", sum(p.numel() for p in backbone_params))
print(" - head lr:", optimizer.param_groups[1]["lr"], "n_params:", sum(p.numel() for p in head_params))

In [ ]:
def masked_macro_auroc(probs_np, targets_np, masks_np):

    per_label_auc = []
    C = targets_np.shape[1]
    for c in range(C):
        valid = masks_np[:, c] == 1
        y_true = targets_np[valid, c]
        y_score = probs_np[valid, c]
        if len(np.unique(y_true)) < 2:
            per_label_auc.append(np.nan)
        else:
            per_label_auc.append(roc_auc_score(y_true, y_score))
    return float(np.nanmean(per_label_auc)), per_label_auc

def evaluate_val(model, loader, device):
    model.eval()
    all_probs, all_targets, all_masks = [], [], []

    with torch.no_grad():
        for xb, yb, mb in tqdm(loader, desc="eval", leave=False):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            mb = mb.to(device, non_blocking=True)

            logits = model(xb)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.detach().cpu())
            all_targets.append(yb.detach().cpu())
            all_masks.append(mb.detach().cpu())

    probs_np = torch.cat(all_probs, dim=0).numpy()
    targets_np = torch.cat(all_targets, dim=0).numpy()
    masks_np = torch.cat(all_masks, dim=0).numpy()

    macro_auc, per_label_auc = masked_macro_auroc(probs_np, targets_np, masks_np)
    return macro_auc, per_label_auc

In [ ]:
def train_masked(
    model, train_loader, val_loader, optimizer, device, pos_weight,
    epochs=3
):
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()


        model.train()
        train_loss_sum = 0.0
        train_steps = 0

        for xb, yb, mb in tqdm(train_loader, desc=f"train {epoch}/{epochs}", leave=False):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            mb = mb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = masked_bce_with_logits(logits, yb, mb, pos_weight=pos_weight)
            loss.backward()
            optimizer.step()

            train_loss_sum += float(loss.detach().cpu())
            train_steps += 1

        train_loss = train_loss_sum / max(train_steps, 1)

        model.eval()
        val_loss_sum = 0.0
        val_steps = 0
        with torch.no_grad():
            for xb, yb, mb in tqdm(val_loader, desc=f"val {epoch}/{epochs}", leave=False):
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                mb = mb.to(device, non_blocking=True)

                logits = model(xb)
                vloss = masked_bce_with_logits(logits, yb, mb, pos_weight=pos_weight)

                val_loss_sum += float(vloss.detach().cpu())
                val_steps += 1

        val_loss = val_loss_sum / max(val_steps, 1)


        val_macro_auc, val_per_label_auc = evaluate_val(model, val_loader, device)

        dt_min = (time.time() - t0) / 60
        print(
            f"Epoch {epoch}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_macro_AUROC={val_macro_auc:.4f} | {dt_min:.1f} min"
        )


        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        print("Restored best model. Best val_loss:", best_val_loss)

    return model

In [ ]:
EPOCHS = 3
model = train_masked(model, train_loader, val_loader, optimizer, device, pos_weight, epochs=EPOCHS)

In [ ]:
def evaluate_test_masked(model, test_loader, device, target_labels):
    model.eval()

    all_probs, all_targets, all_masks = [], [], []

    with torch.no_grad():
        for xb, yb, mb in tqdm(test_loader, desc="Test evaluation (masked)"):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            mb = mb.to(device, non_blocking=True)

            logits = model(xb)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.detach().cpu())
            all_targets.append(yb.detach().cpu())
            all_masks.append(mb.detach().cpu())

    probs_np   = torch.cat(all_probs, dim=0).numpy()
    targets_np = torch.cat(all_targets, dim=0).numpy()
    masks_np   = torch.cat(all_masks, dim=0).numpy()

    per_label_auc = []
    for c in range(targets_np.shape[1]):
        valid = masks_np[:, c] > 0.5
        y_true = targets_np[valid, c]
        y_score = probs_np[valid, c]


        if y_true.size == 0 or len(np.unique(y_true)) < 2:
            per_label_auc.append(np.nan)
        else:
            per_label_auc.append(roc_auc_score(y_true, y_score))

    macro_auc = np.nanmean(per_label_auc)

    print("\nTest Macro AUROC (masked):", float(macro_auc))
    print("\nPer-label AUROC (masked):")
    for name, auc in zip(target_labels, per_label_auc):
        print(f"{name:18s}: {auc:.4f}")

    print("\nValid test samples per label:")
    for name, cnt in zip(target_labels, masks_np.sum(axis=0).astype(int)):
        print(f"{name:18s}: {cnt}")

    return float(macro_auc), [float(x) if not np.isnan(x) else None for x in per_label_auc]

test_macro_auc_masked, test_per_label_auc_masked = evaluate_test_masked(
    model, test_loader, device, TARGET_LABELS
)